<a href="https://colab.research.google.com/github/psh135230-design/BioAI_KU_2026/blob/main/%EB%B0%94%EC%9D%B4%EC%98%A4%EC%9D%B8%EA%B3%B5%EC%A7%80%EB%8A%A5_week2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
%pwd

'/content'

In [28]:
import subprocess
import sys

scanpy_stack = ["scanpy==1.11.5", "anndata==0.12.19"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *scanpy_stack])
print("Installed:", ", ".join(scanpy_stack))

Installed: scanpy==1.11.5, anndata==0.12.19


In [29]:
from pathlib import Path
import importlib.metadata as metadata
import warnings
import traceback

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

warnings.filterwarnings("ignore", category=FutureWarning)
sc.settings.set_figure_params(dpi=110, facecolor="white", frameon=False)
print("scanpy", metadata.version("scanpy"))
print("anndata", metadata.version("anndata"))

scanpy 1.11.5
anndata 0.12.19


In [30]:
def convert_geodata_to_h5ad(data_dir, prefix, metadata_file, output_file):
    """
    10x MTX 세트와 메타데이터 CSV를 읽어 안정적인 .h5ad 파일로 변환하는 범용 함수
    """
    print(f"[{output_file}] 변환 시작...")

    # 1. 10x MTX 로드 (X 행렬 및 var 데이터 생성)
    adata = sc.read_10x_mtx(
        path=data_dir,
        prefix=prefix,
        var_names="gene_symbols",
        make_unique=True
    )

    # Sparse CSR 포맷 보장 (메모리 및 저장 공간 최적화)
    if not sparse.isspmatrix_csr(adata.X): # Changed from sc.sparse.isspmatrix_csr
        adata.X = sparse.csr_matrix(adata.X)

    # 2. 메타데이터 로드 및 중복 인덱스 제거
    metadata = pd.read_csv(metadata_file, index_col=0)
    metadata = metadata[~metadata.index.duplicated(keep='first')]

    # 3. AnnData의 obs_names에 맞춰 메타데이터 재인덱싱 (행 수 및 바코드 맞춤)
    adata.obs = metadata.reindex(adata.obs_names)

    # 4. h5ad 저장 오류 방지를 위해 비범주형 열을 문자열(str)로 변환
    for col in adata.obs.columns:
        if not pd.api.types.is_categorical_dtype(adata.obs[col]):
            adata.obs[col] = adata.obs[col].astype(str)

    # 5. h5ad 파일 저장
    adata.write_h5ad(output_file)
    print(f"✅ {output_file} 생성 완료!\n")
    return adata

In [31]:
dataset_configs = [
    {
        "data_dir": ".",
        "prefix": "GSM9621184_lib1_",
        "metadata_file": "GSM9621184_Cell_Metadata.csv.gz",
        "output_file": "GSE326061.sparse.h5ad"
    },
    {
        "data_dir": ".",
        "prefix": "GSM9617712_lib2_",
        "metadata_file": "GSM9617712_Cell_Metadata.csv.gz",
        "output_file": "GSE325953.sparse.h5ad"
    }
    # 추후 새로운 데이터셋이 생기면 위와 같이 디셔너리로 추가
]

In [22]:
for config in dataset_configs:
    try:
        convert_geodata_to_h5ad(
            data_dir=config["data_dir"],
            prefix=config["prefix"],
            metadata_file=config["metadata_file"],
            output_file=config["output_file"]
        )
    except KeyboardInterrupt:
        print(f"❌ [{config['output_file']}] 처리 중 수동으로 중단되었습니다.\n")
        raise # KeyboardInterrupt를 다시 발생시켜 실행을 중단합니다.
    except Exception as e:
        print(f"❌ [{config['output_file']}] 처리 중 에러 발생: {type(e).__name__}: {e}\n")
        traceback.print_exc() # Print the full traceback

[GSE326061.sparse.h5ad] 변환 시작...


/tmp/ipykernel_24012/790562583.py:28: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if not pd.api.types.is_categorical_dtype(adata.obs[col]):
/tmp/ipykernel_24012/790562583.py:28: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if not pd.api.types.is_categorical_dtype(adata.obs[col]):
/tmp/ipykernel_24012/790562583.py:28: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if not pd.api.types.is_categorical_dtype(adata.obs[col]):


✅ GSE326061.sparse.h5ad 생성 완료!

[GSE325953.sparse.h5ad] 변환 시작...


/tmp/ipykernel_24012/790562583.py:28: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if not pd.api.types.is_categorical_dtype(adata.obs[col]):
/tmp/ipykernel_24012/790562583.py:28: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if not pd.api.types.is_categorical_dtype(adata.obs[col]):
/tmp/ipykernel_24012/790562583.py:28: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if not pd.api.types.is_categorical_dtype(adata.obs[col]):


✅ GSE325953.sparse.h5ad 생성 완료!



In [23]:
processed_h5ad = Path("GSE326061.sparse.h5ad")

if processed_h5ad.exists():
    adata_GSE326061 = sc.read_h5ad(processed_h5ad)
    dataset_status = "loaded_local_GSE326061_sparse_h5ad"
else:
    adata_GSE326061 = sc.read_h5ad(processed_h5ad)
    dataset_status = "GSE326061.sparse.h5ad 생성 완료!"

print(dataset_status)
print(adata_GSE326061)

loaded_local_GSE326061_sparse_h5ad
AnnData object with n_obs × n_vars = 2965363 × 72302
    obs: 'treatment', 'cell_type', 'cluster'
    var: 'gene_ids', 'feature_types'


In [24]:
processed_h5ad = Path("GSE325953.sparse.h5ad")

if processed_h5ad.exists():
    adata_GSE325953 = sc.read_h5ad(processed_h5ad)
    dataset_status = "loaded_local_GSE325953_sparse_h5ad"
else:
    # 생성 코드가 앞서 실행되어 파일이 생성된다고 가정
    adata_GSE325953 = sc.read_h5ad(processed_h5ad)
    dataset_status = "GSE325953.sparse.h5ad 생성 완료!"

print(dataset_status)
print(adata_GSE325953)

loaded_local_GSE325953_sparse_h5ad
AnnData object with n_obs × n_vars = 2979617 × 72302
    obs: 'treatment', 'cell_type', 'cluster'
    var: 'gene_ids', 'feature_types'


In [25]:
adatas = [
    ("GSE326061", adata_GSE326061),
    ("GSE325953", adata_GSE325953)
]
summary_tables = []

In [26]:
for name, adata in adatas:
    df = pd.DataFrame(
        {
            "dataset": name,
            "slot": ["X", "obs", "var", "obsm", "layers", "uns", "raw"],
            "contents": [
                "cell-by-gene expression matrix",
                "cell metadata",
                "gene metadata",
                "cell embeddings",
                "alternative matrices",
                "analysis metadata",
                "optional frozen expression reference",
            ],
            "present_or_shape": [
                str(adata.X.shape),
                str(adata.obs.shape),
                str(adata.var.shape),
                ", ".join(adata.obsm.keys()) or "none",
                ", ".join(adata.layers.keys()) or "none",
                f"{len(adata.uns)} keys",
                "yes" if adata.raw is not None else "no",
            ],
        }
    )
    summary_tables.append(df)

# 3. 하나의 데이터프레임으로 결합
slot_summary = pd.concat(summary_tables, ignore_index=True)
slot_summary

,dataset,slot,contents,present_or_shape
0,GSE326061,X,cell-by-gene expression matrix,"(2965363, 72302)"
1,GSE326061,obs,cell metadata,"(2965363, 3)"
2,GSE326061,var,gene metadata,"(72302, 2)"
3,GSE326061,obsm,cell embeddings,none
4,GSE326061,layers,alternative matrices,none
5,GSE326061,uns,analysis metadata,0 keys
6,GSE326061,raw,optional frozen expression reference,no
7,GSE325953,X,cell-by-gene expression matrix,"(2979617, 72302)"
8,GSE325953,obs,cell metadata,"(2979617, 3)"
9,GSE325953,var,gene metadata,"(72302, 2)"
